# LAB | Imbalanced

**Load the data**

In this challenge, we will be working with Credit Card Fraud dataset.

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv

Metadata

- **distance_from_home:** the distance from home where the transaction happened.
- **distance_from_last_transaction:** the distance from last transaction happened.
- **ratio_to_median_purchase_price:** Ratio of purchased price transaction to median purchase price.
- **repeat_retailer:** Is the transaction happened from same retailer.
- **used_chip:** Is the transaction through chip (credit card).
- **used_pin_number:** Is the transaction happened by using PIN number.
- **online_order:** Is the transaction an online order.
- **fraud:** Is the transaction fraudulent. **0=legit** -  **1=fraud**


In [ ]:
#Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, classification_report, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

In [ ]:
fraud = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/card_transdata.csv")
fraud.head()

**Steps:**

- **1.** What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?
- **2.** Train a LogisticRegression.
- **3.** Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.
- **4.** Run **Oversample** in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model? 
- **5.** Now, run **Undersample** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?
- **6.** Finally, run **SMOTE** in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model? 

In [ ]:
# 1. What is the distribution of our target variable? Can we say we're dealing with an imbalanced dataset?

In [ ]:
distribution = fraud["fraud"].value_counts()
display(distribution)
distribution.plot(kind="bar")
plt.show()

In [ ]:
print("Yes, we can talk about an imbalanced dataset, as the value 1 for fraud has a significantly lower occurence (8,7%).")

In [ ]:
# 2. Train a LogisticRegression.

In [ ]:
features = fraud.drop(columns = ["fraud"])
target = fraud["fraud"]

X_train, X_test, y_train, y_test = train_test_split(features, target,random_state=0)

In [ ]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
log_reg = LogisticRegression()

In [ ]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled_df  = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

In [ ]:
log_reg.fit(X_train_scaled_df, y_train)

In [ ]:
log_reg.score(X_test_scaled_df, y_test)

In [ ]:
y_pred_test_log = log_reg.predict(X_test_scaled_df)
print(classification_report(y_pred = y_pred_test_log, y_true = y_test))

In [ ]:
# 3. Evaluate your model. Take in consideration class importance, and evaluate it by selection the correct metric.

In [ ]:
print("On the first sight the results look really good, because accuracy is 97%. But this is due to the fact that it's very likely to find no fraud in the dataset, so predicting mainly no fraud will lead to good results. The main aim is to find fraud. Here we have to check the Recall, that is the ratio of how many frauds were actually detected, and Precision, which states how many of the detected frauds were actually frauds. The combination of these both metrics is the F1-score, which only performs with 72%. Especially the Recall is rather low, with only 60% of frauds detected.")

In [ ]:
# 4. Run Oversample in order to balance our target variable and repeat the steps above, now with balanced data. Does it improve the performance of our model?

In [ ]:
train = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)

In [ ]:
train["fraud"] = y_train.values

In [ ]:
fraud_true = train[train["fraud"] == 1]
no_fraud = train[train["fraud"] == 0]

In [ ]:
len(fraud_true),len(no_fraud)

In [ ]:
yes_oversampled = resample(fraud_true,
                                    replace=True,
                                    n_samples = len(no_fraud),
                                    random_state=0)

In [ ]:
train_over = pd.concat([yes_oversampled, no_fraud])
train_over

In [ ]:
X_train_over = train_over.drop(columns = ["fraud"])
y_train_over = train_over["fraud"]

In [ ]:
log_reg = LogisticRegression()
log_reg.fit(X_train_over, y_train_over)

In [ ]:
y_pred_test_log = log_reg.predict(X_test_scaled_df)
print(classification_report(y_pred = y_pred_test_log, y_true = y_test))

In [ ]:
print("The Oversampling did improve the detection of fraud. The Recall is at 95%, which means that 95% of fraud was detected. But now it looks like there were 'too many' frauds detected, beacuse only 57% of detected frauds were actual frauds. This improvement of Recall and worsening of Precision leads to equal F1-score of 71%.")

In [ ]:
# 5. Now, run Undersample in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?

In [ ]:
no_undersampled = resample(no_fraud,
                                    replace=False,
                                    n_samples = len(fraud_true),
                                    random_state=0)
no_undersampled

In [ ]:
train_under = pd.concat([no_undersampled, fraud_true])
train_under

In [ ]:
X_train_under = train_under.drop(columns = ["fraud"])
y_train_under = train_under["fraud"]

In [ ]:
log_reg = LogisticRegression()
log_reg.fit(X_train_under, y_train_under)

In [ ]:
y_pred_test_log = log_reg.predict(X_test_scaled_df)
print(classification_report(y_pred = y_pred_test_log, y_true = y_test))

In [ ]:
print("There seems to be no difference in performance of Oversampling and Undersampling.")

In [ ]:
# 6. Finally, run SMOTE in order to balance our target variable and repeat the steps above (1-3), now with balanced data. Does it improve the performance of our model?

In [ ]:
from imblearn.over_sampling import SMOTE

In [ ]:
sm = SMOTE(random_state = 1,sampling_strategy=1.0)

In [ ]:
X_train_sm, y_train_sm =  sm.fit_resample(X_train_scaled_df,y_train) # .fit() doesn't exists!!!

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_sm, y_train_sm)

In [ ]:
y_pred_test_log = log_reg.predict(X_test_scaled_df)
print(classification_report(y_pred = y_pred_test_log, y_true = y_test))

In [ ]:
print("I'm not sure, if I did something wrong, but all the balancing methods show the same results.")